In [3]:
import pennylane as qml
from scipy.optimize import minimize
import numpy as np
from tensorflow.keras.datasets import mnist
from matplotlib import pyplot as plt
from scipy.linalg import expm

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train_ones = x_train[y_train == 2]
x_train_ones_32 = np.pad(x_train_ones, ((0, 0), (2, 2), (2, 2)), mode='constant', constant_values=0)
Image = x_train_ones_32[0]


In [4]:
Img = np.cov(Image/255.0)
O = expm(1j * Img)

In [5]:
def int_to_bin(n, num_bits)-> tuple:
        return tuple(int(x) for x in format(n, f'0{num_bits}b'))

In [14]:
n_qubits = 10
q_qubits = 5
l_qubits = 5
dev = qml.device("default.qubit", wires=n_qubits) #, shots=1000
@qml.qnode(dev)
def ImageEigen(theta, image, control_values=True):
    
    # Definition of the Proyector
    P1 = np.array([[0,0],[0,1]])

    # Optional Hadamards section
    if control_values:
        for i in range(l_qubits,n_qubits):
            qml.Hadamard(wires=i)
    # Mottonen Ansatz
    c = 2**q_qubits -2
    for k,i in enumerate(range(q_qubits-1, -1, -1)):
        for j in range(2**k-1,-1,-1):
            if k>0:
                qml.ctrl(qml.adjoint(qml.RY(theta[c], wires=k+l_qubits)), control=range(l_qubits,k+l_qubits), control_values=int_to_bin(j,k))
            else:
                qml.adjoint(qml.RY(theta[c], wires=k+q_qubits))
            c = c - 1
    
    qml.Barrier()
    
    for i in range(5):
        qml.Hadamard(wires=i)
    qml.ControlledSequence(qml.QubitUnitary(image, wires=range(5,10)),control=range(5))
    qml.adjoint(qml.QFT(wires=range(5)))
    H = np.kron(P1,np.eye(2**(q_qubits -1)))
    return qml.expval(qml.Hermitian(H,wires=range(5)))

In [15]:
circuit = qml.QNode(ImageEigen,dev)
theta = np.random.uniform(0,2*np.pi,2**q_qubits -1)
l = circuit(theta,O,control_values=True)
l

np.float64(0.0049688643452917725)

In [17]:
def simple_cost(weights,O):
    """Computes the cost over the provided features and labels"""
    preds = circuit(weights, O)
    return (1.00-preds) ** 2

In [18]:
theta_0 = np.zeros(shape=(2**q_qubits -1))
theta_0

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [19]:
simple_cost(theta_0,O)

np.float64(0.9792149822570294)

In [20]:
result = minimize(
    simple_cost, theta_0, O, method="Nelder-Mead", options={"xatol": 1e-8, "disp": True}
)

/tmp/ipykernel_7746/2824035569.py:1: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
